# NLP Feature Extraction — Pure PySpark
### HashingTF | IDF | TF-IDF | Keyword Search | Word2Vec
**Dataset:** `shakespeare.txt` | Google Colab / Jupyter | Pure Spark

## Step 0 — Spark Setup (Google Colab)

In [3]:
import os, sys, ssl, urllib.request

# ── Download correct 64-bit winutils ─────────────────────────────────────────
os.makedirs('hadoop/bin', exist_ok=True)
ctx  = ssl._create_unverified_context()
base = 'https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.6/bin/'

for f in ['winutils.exe', 'hadoop.dll']:
    dest = f'hadoop/bin/{f}'
    urllib.request.urlretrieve(base + f, dest)
    print(f'✅ {f} → {os.path.getsize(dest)} bytes')

# ── Environment variables — MUST be absolute path ────────────────────────────
os.environ['JAVA_HOME']      = r'C:\Users\yrghimire\.jdks\corretto-1.8.0_482'
os.environ['HADOOP_HOME']    = os.path.abspath('hadoop')   # ✅ absolute path
os.environ['PATH']           = os.path.abspath('hadoop/bin') + ';' + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.makedirs(r'C:\tmp\hive', exist_ok=True)                 # ✅ required by Hadoop

print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField,
                                StringType, DoubleType, IntegerType)

try:
    SparkContext.getOrCreate().stop()
except:
    pass

conf = (SparkConf()
        .setAppName('SparkSQL_MCar')
        .setMaster('local[*]')
        .set('spark.driver.host',            'localhost')
        .set('spark.driver.memory',          '4g')
        .set('spark.sql.codegen.wholeStage', 'false'))

sc    = SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()
sc.setLogLevel('ERROR')

os.makedirs('output/SparkSQL_MCar', exist_ok=True)
print('✅ Spark ready:', spark.version)


✅ winutils.exe → 119296 bytes
✅ hadoop.dll → 78848 bytes
HADOOP_HOME: c:\Users\yrghimire\Chapter\spring_2026\cse817\bigdata_act\chap2\hadoop
✅ Spark ready: 3.5.3


## Step 1 — Download shakespeare.txt

In [ ]:
# import urllib.request

# url  = 'https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt'
# path = 'shakespeare.txt'

# if not os.path.exists(path):
#     urllib.request.urlretrieve(url, path)
#     print(f'✅ Downloaded: {os.path.getsize(path):,} bytes')
# else:
#     print(f'✅ Already exists: {os.path.getsize(path):,} bytes')

✅ Downloaded: 5,458,199 bytes


## Step 2 — Load into Spark DataFrame (one sentence per row)

In [5]:
# Read file — each line = one document/sentence
raw_df = (spark.read
          .option('wholetext', 'false')
          .text('Dataset/shakespeare.txt')
          .toDF('sentence')
          .filter(F.col('sentence') != '')        # remove blank lines
          .filter(F.length(F.col('sentence')) > 10)  # remove short lines
          .withColumn('sentence', F.lower(F.col('sentence')))  # lowercase
          .withColumn('doc_id', F.monotonically_increasing_id())
         )

print(f'Total lines/documents: {raw_df.count()}')
raw_df.show(10, truncate=80)

Total lines/documents: 113523
+-------------------------------------------------------------------+------+
|                                                           sentence|doc_id|
+-------------------------------------------------------------------+------+
|   this is the 100th etext file presented by project gutenberg, and|     0|
|   is presented in cooperation with world library, inc., from their|     1|
|   library of the future and shakespeare cdroms.  project gutenberg|     2|
|   often releases etexts that are not placed in the public domain!!|     3|
|                                                        shakespeare|     4|
|   *this etext has certain copyright implications you should read!*|     5|
|         <<this electronic version of the complete works of william|     6|
|  shakespeare is copyright 1990-1993 by world library, inc., and is|     7|
|provided by project gutenberg etext of illinois benedictine college|     8|
|    with permission.  electronic and machine 

## Step 3 — Tokenize + Remove Stop Words

In [7]:

from pyspark.ml.feature import (Tokenizer, StopWordsRemover,
                                 HashingTF, IDF, Word2Vec, Word2VecModel)
from pyspark.ml import Pipeline


# Step 1: Tokenizer — split sentence into words
tokenizer = Tokenizer(inputCol='sentence', outputCol='words_raw')

# Step 2: Remove stop words (the, is, at, which...)
remover   = StopWordsRemover(inputCol='words_raw', outputCol='words')

# Apply both steps
tokenized_df = tokenizer.transform(raw_df)
clean_df     = remover.transform(tokenized_df)

print('Sample tokenized + cleaned words:')
clean_df.select('sentence', 'words_raw', 'words').show(5, truncate=60)

Sample tokenized + cleaned words:
+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|                                                    sentence|                                                   words_raw|                                                       words|
+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|this is the 100th etext file presented by project gutenbe...|[this, is, the, 100th, etext, file, presented, by, projec...|        [100th, etext, file, presented, project, gutenberg,]|
|is presented in cooperation with world library, inc., fro...|[is, presented, in, cooperation, with, world, library,, i...|            [presented, cooperation, world, library,, inc.,]|
|library of the future and shakespeare cd

## Step 4 — (a) HashingTF: Term Frequency

In [8]:
# HashingTF — maps words to fixed-size feature vector using hashing
hashingTF = HashingTF(
    inputCol='words',
    outputCol='tf_features',
    numFeatures=10000          # vocabulary size (hash buckets)
)

tf_df = hashingTF.transform(clean_df)

print('TF (Term Frequency) features:')
tf_df.select('doc_id', 'sentence', 'tf_features').show(5, truncate=60)

# Show TF vector details for first document
print('\nTF vector for first document (sparse format):')
tf_df.select('tf_features').limit(1).show(truncate=False)

TF (Term Frequency) features:
+------+------------------------------------------------------------+------------------------------------------------------------+
|doc_id|                                                    sentence|                                                 tf_features|
+------+------------------------------------------------------------+------------------------------------------------------------+
|     0|this is the 100th etext file presented by project gutenbe...|(10000,[2230,2313,4115,5108,6716,9488],[1.0,1.0,1.0,1.0,1...|
|     1|is presented in cooperation with world library, inc., fro...|    (10000,[1738,4115,6643,8588,9360],[1.0,1.0,1.0,1.0,1.0])|
|     2|library of the future and shakespeare cdroms.  project gu...|(10000,[2230,3372,3808,4236,4803,5635,6488],[1.0,1.0,1.0,...|
|     3|often releases etexts that are not placed in the public d...|(10000,[7,515,1030,1770,5453,8792],[1.0,1.0,1.0,1.0,1.0,1...|
|     4|                                             

## Step 5 — (a) IDF: Inverse Document Frequency

In [9]:
# IDF — down-weights terms that appear in many documents
idf       = IDF(inputCol='tf_features', outputCol='tfidf_features', minDocFreq=2)
idf_model = idf.fit(tf_df)
tfidf_df  = idf_model.transform(tf_df)

print('TF-IDF features:')
tfidf_df.select('doc_id', 'sentence', 'tfidf_features').show(5, truncate=60)

# IDF values (log scale — higher = rarer word = more discriminative)
import numpy as np
idf_values = idf_model.idf.toArray()
print(f'\nIDF vector size    : {len(idf_values)}')
print(f'Max IDF value      : {idf_values.max():.4f}  (rarest term)')
print(f'Min IDF value      : {idf_values[idf_values > 0].min():.4f}  (most common term)')
print(f'Non-zero IDF values: {(idf_values > 0).sum()}')

TF-IDF features:
+------+------------------------------------------------------------+------------------------------------------------------------+
|doc_id|                                                    sentence|                                              tfidf_features|
+------+------------------------------------------------------------+------------------------------------------------------------+
|     0|this is the 100th etext file presented by project gutenbe...|(10000,[2230,2313,4115,5108,6716,9488],[6.102435280276359...|
|     1|is presented in cooperation with world library, inc., fro...|(10000,[1738,4115,6643,8588,9360],[8.307565037119693,6.71...|
|     2|library of the future and shakespeare cdroms.  project gu...|(10000,[2230,3372,3808,4236,4803,5635,6488],[6.1024352802...|
|     3|often releases etexts that are not placed in the public d...|(10000,[7,515,1030,1770,5453,8792],[6.764572224093745,9.8...|
|     4|                                                 shakespea

## Step 6 — (a) Document Frequency (DF) per Word

In [10]:
# Compute DF — how many documents each word appears in
# Explode words → count distinct doc_id per word
df_counts = (clean_df
    .select('doc_id', F.explode(F.col('words')).alias('word'))
    .filter(F.length(F.col('word')) > 2)           # skip tiny words
    .groupBy('word')
    .agg(F.countDistinct('doc_id').alias('doc_freq'),
         F.count('word').alias('total_freq'))
    .orderBy(F.col('doc_freq').desc())
)

print('Top 20 words by Document Frequency (DF):')
df_counts.show(20, truncate=False)

# Register for SQL queries
df_counts.createOrReplaceTempView('word_freq')
tfidf_df.createOrReplaceTempView('tfidf_docs')

Top 20 words by Document Frequency (DF):
+-----+--------+----------+
|word |doc_freq|total_freq|
+-----+--------+----------+
|thou |4771    |5138      |
|thy  |3745    |4028      |
|shall|3417    |3473      |
|good |2478    |2560      |
|let  |2055    |2089      |
|enter|2008    |2011      |
|hath |1880    |1902      |
|thee |1748    |1794      |
|may  |1735    |1747      |
|king |1674    |1697      |
|upon |1660    |1664      |
|like |1622    |1642      |
|make |1581    |1596      |
|you, |1436    |1479      |
|one  |1435    |1486      |
|must |1421    |1449      |
|know |1379    |1402      |
|'tis |1337    |1367      |
|come |1332    |1358      |
|give |1278    |1286      |
+-----+--------+----------+
only showing top 20 rows



## Step 7 — (a) TF-IDF Summary Table via Spark SQL

In [11]:
# Summary stats via Spark SQL
print('=== TF-IDF Summary ===')
spark.sql("""
    SELECT
        COUNT(*)       AS total_documents,
        COUNT(DISTINCT sentence) AS unique_sentences
    FROM tfidf_docs
""").show()

print('=== Top 20 Most Frequent Words (DF) ===')
spark.sql("""
    SELECT word, doc_freq, total_freq
    FROM   word_freq
    ORDER  BY doc_freq DESC
    LIMIT  20
""").show(truncate=False)

print('=== Rarest Words (DF = 2) ===')
spark.sql("""
    SELECT word, doc_freq, total_freq
    FROM   word_freq
    WHERE  doc_freq = 2
    ORDER  BY total_freq DESC
    LIMIT  20
""").show(truncate=False)

=== TF-IDF Summary ===
+---------------+----------------+
|total_documents|unique_sentences|
+---------------+----------------+
|         113523|          110757|
+---------------+----------------+

=== Top 20 Most Frequent Words (DF) ===
+-----+--------+----------+
|word |doc_freq|total_freq|
+-----+--------+----------+
|thou |4771    |5138      |
|thy  |3745    |4028      |
|shall|3417    |3473      |
|good |2478    |2560      |
|let  |2055    |2089      |
|enter|2008    |2011      |
|hath |1880    |1902      |
|thee |1748    |1794      |
|may  |1735    |1747      |
|king |1674    |1697      |
|upon |1660    |1664      |
|like |1622    |1642      |
|make |1581    |1596      |
|you, |1436    |1479      |
|one  |1435    |1486      |
|must |1421    |1449      |
|know |1379    |1402      |
|'tis |1337    |1367      |
|come |1332    |1358      |
|give |1278    |1286      |
+-----+--------+----------+

=== Rarest Words (DF = 2) ===
+---------+--------+----------+
|word     |doc_freq|total_

## Step 8 — (a) Keyword Search in Documents using TF-IDF

In [12]:
# ── Search for a specific keyword in documents ────────────────────────────────
KEYWORD = 'hamlet'    # ← change this to any word you want to search

print(f'=== Searching for keyword: "{KEYWORD}" ===')

# Method 1: DataFrame API — find documents containing the keyword
keyword_docs = (clean_df
    .filter(F.array_contains(F.col('words'), KEYWORD))
    .select('doc_id', 'sentence')
    .orderBy('doc_id')
)
print(f'Documents containing "{KEYWORD}": {keyword_docs.count()}')
keyword_docs.show(10, truncate=80)

# Method 2: Spark SQL search
print(f'\n=== SQL Search: sentences containing "{KEYWORD}" ===')
spark.sql(f"""
    SELECT doc_id, sentence
    FROM   tfidf_docs
    WHERE  sentence LIKE '%{KEYWORD}%'
    ORDER  BY doc_id
    LIMIT  10
""").show(truncate=80)

# Keyword stats across document
print(f'\n=== "{KEYWORD}" frequency stats ===')
spark.sql(f"""
    SELECT word, doc_freq, total_freq
    FROM   word_freq
    WHERE  word = '{KEYWORD}'
""").show(truncate=False)

=== Searching for keyword: "hamlet" ===
Documents containing "hamlet": 27
+------+-----------------------------------------------------+
|doc_id|                                             sentence|
+------+-----------------------------------------------------+
| 21860|     dar'd to the combat; in which our valiant hamlet|
| 21962|  king. though yet of hamlet our dear brother's death|
| 22087|            this gentle and unforc'd accord of hamlet|
| 22701|                  and what so poor a man as hamlet is|
| 22893|           and bring these gentlemen where hamlet is.|
| 22984|                 queen. came this from hamlet to her?|
| 23013|           'lord hamlet is a prince, out of thy star.|
| 23497|          for we have closely sent for hamlet hither,|
| 23651|          you need not tell us what lord hamlet said.|
| 23663|               enter hamlet and three of the players.|
+------+-----------------------------------------------------+
only showing top 10 rows


=== SQL Search: s

## Step 9 — Save TF-IDF Results

In [13]:
# Save word DF/IDF table
out_df = 'file:///' + os.path.abspath('output/NLP_Shakespeare/word_doc_freq').replace('\\','/')
(df_counts
 .coalesce(1)
 .write.format('csv')
 .mode('overwrite')
 .option('header', 'true')
 .save(out_df))
print('✅ Word DF saved')

# Save keyword search results
out_kw = 'file:///' + os.path.abspath('output/NLP_Shakespeare/keyword_search').replace('\\','/')
(keyword_docs
 .coalesce(1)
 .write.format('csv')
 .mode('overwrite')
 .option('header', 'true')
 .save(out_kw))
print('✅ Keyword results saved')

✅ Word DF saved
✅ Keyword results saved


## Step 10 — (b) Word2Vec: Train Model

In [14]:
# Word2Vec — learns dense vector representation for each word
word2vec = Word2Vec(
    inputCol='words',
    outputCol='w2v_features',
    vectorSize=100,      # embedding dimension
    minCount=3,          # ignore words appearing < 3 times
    numPartitions=4,
    seed=42
)

print('Training Word2Vec model...')
w2v_model = word2vec.fit(clean_df)
print('✅ Word2Vec trained')
print(f'Vocabulary size: {len(w2v_model.getVectors().collect())}')

Training Word2Vec model...
✅ Word2Vec trained
Vocabulary size: 19796


## Step 11 — (b) Get Word Vectors

In [15]:
# Get the learned word vectors as a Spark DataFrame
word_vectors_df = w2v_model.getVectors()

print(f'Word vectors DataFrame:')
print(f'Total words in vocabulary: {word_vectors_df.count()}')
word_vectors_df.show(10, truncate=60)

# Register as SQL table
word_vectors_df.createOrReplaceTempView('word_vectors')

# Show vector for a specific word
print('\nVector for word "king":')
spark.sql("""
    SELECT word, vector
    FROM   word_vectors
    WHERE  word = 'king'
""").show(truncate=False)

Word vectors DataFrame:
Total words in vocabulary: 19796
+---------+------------------------------------------------------------+
|     word|                                                      vector|
+---------+------------------------------------------------------------+
|professed|[7.489470299333334E-4,-0.0029319149907678366,-0.002179109...|
|    come?|[-0.005357897840440273,0.02476561814546585,0.211782664060...|
| herself.|[0.056756675243377686,0.04229232668876648,0.0997561961412...|
|   corse;|[-0.009126785211265087,3.581837227102369E-4,0.08090674877...|
|     ice;|[-0.0017099056858569384,0.0089957844465971,0.040314815938...|
| incident|[-0.0017539304681122303,0.014395438134670258,0.0231634397...|
|   drums.|[0.0395999401807785,-0.009622765704989433,0.1217364966869...|
|  serious|[0.03571503236889839,0.021513333544135094,0.1626191884279...|
| youthful|[0.027533967047929764,-0.01800525188446045,0.120545580983...|
| sinister|[0.0030248104594647884,0.00428366381675005,0.01386795006

## Step 12 — (b) Find Similar Words (Synonyms)

In [16]:
# Find top N similar words for given keywords
keywords = ['king', 'love', 'death', 'sword', 'romeo']

all_similar = []

for word in keywords:
    try:
        # findSynonyms returns a Spark DataFrame — pure Spark operation
        similar_df = (w2v_model
                      .findSynonyms(word, 10)
                      .withColumn('query_word', F.lit(word))
                      .select('query_word', 'word', 'similarity')
                     )
        print(f'\n=== Top 10 words similar to "{word}" ===')
        similar_df.show(truncate=False)
        all_similar.append(similar_df)
    except Exception as e:
        print(f'  "{word}" not in vocabulary: {e}')


=== Top 10 words similar to "king" ===
+----------+-----------+------------------+
|query_word|word       |similarity        |
+----------+-----------+------------------+
|king      |king,      |0.6517207026481628|
|king      |queen      |0.6130416989326477|
|king      |york.      |0.5985981822013855|
|king      |son        |0.5927374958992004|
|king      |york       |0.5901824235916138|
|king      |advancement|0.5900065302848816|
|king      |warwick.   |0.5667967796325684|
|king      |salisbury  |0.561914324760437 |
|king      |oxford     |0.557249128818512 |
|king      |[he        |0.553310751914978 |
+----------+-----------+------------------+


=== Top 10 words similar to "love" ===
+----------+-----+------------------+
|query_word|word |similarity        |
+----------+-----+------------------+
|love      |know |0.7749934196472168|
|love      |yet  |0.7411361932754517|
|love      |much |0.7306493520736694|
|love      |love,|0.7163597941398621|
|love      |never|0.6973721981048584|

## Step 13 — (b) Word2Vec: Transform Documents to Vectors

In [17]:
# Transform each document into a single averaged Word2Vec vector
w2v_docs_df = w2v_model.transform(clean_df)

print('Document vectors (averaged Word2Vec):')
w2v_docs_df.select('doc_id', 'sentence', 'w2v_features').show(5, truncate=60)

# Similarity between two specific sentences using Spark SQL
w2v_docs_df.createOrReplaceTempView('w2v_docs')

print('\n=== Word2Vec Doc Summary via Spark SQL ===')
spark.sql("""
    SELECT COUNT(*) AS total_docs
    FROM   w2v_docs
    WHERE  w2v_features IS NOT NULL
""").show()

Document vectors (averaged Word2Vec):
+------+------------------------------------------------------------+------------------------------------------------------------+
|doc_id|                                                    sentence|                                                w2v_features|
+------+------------------------------------------------------------+------------------------------------------------------------+
|     0|this is the 100th etext file presented by project gutenbe...|[0.010940385051071644,0.531867263217767,0.140184431647260...|
|     1|is presented in cooperation with world library, inc., fro...|[-0.1402889657765627,0.0426082331687212,0.293891494721174...|
|     2|library of the future and shakespeare cdroms.  project gu...|[0.06670242528031979,0.5659419485766972,0.116395838093012...|
|     3|often releases etexts that are not placed in the public d...|[0.013173131805766994,0.010902648869281013,0.087567065687...|
|     4|                                     

## Step 14 — Save Word2Vec Results

In [18]:
from pyspark.sql import Row

# Save word vectors (word + vector as string)
out_wv = 'file:///' + os.path.abspath('output/NLP_Shakespeare/word_vectors').replace('\\','/')
(word_vectors_df
 .withColumn('vector_str', F.col('vector').cast(StringType()))
 .select('word', 'vector_str')
 .coalesce(1)
 .write.format('csv')
 .mode('overwrite')
 .option('header', 'true')
 .save(out_wv))
print('✅ Word vectors saved')

# Save all similar words results
if all_similar:
    from functools import reduce
    combined = reduce(lambda a, b: a.union(b), all_similar)
    out_sim  = 'file:///' + os.path.abspath('output/NLP_Shakespeare/similar_words').replace('\\','/')
    (combined
     .coalesce(1)
     .write.format('csv')
     .mode('overwrite')
     .option('header', 'true')
     .save(out_sim))
    print('✅ Similar words saved')

✅ Word vectors saved
✅ Similar words saved


## Step 15 — Final Summary & Stop Spark

In [19]:
print('='*60)
print('   NLP FEATURE EXTRACTION — Shakespeare — SUMMARY')
print('='*60)
print(f'  Total documents     : {raw_df.count()}')
print(f'  Vocabulary (DF)     : {df_counts.count()} unique words')
print(f'  TF-IDF features     : 10000 hash buckets')
print(f'  Word2Vec vocab      : {word_vectors_df.count()} words')
print(f'  Word2Vec vector dim : 100')
print(f'  Keyword searched    : "{KEYWORD}" → {keyword_docs.count()} docs')
print()
print('  Output saved to: output/NLP_Shakespeare/')
print('    ├── word_doc_freq/      (DF per word)')
print('    ├── keyword_search/     (documents with keyword)')
print('    ├── word_vectors/       (Word2Vec embeddings)')
print('    └── similar_words/      (Word2Vec similarities)')
print('='*60)

spark.stop()
print('\n✅ Spark stopped. All done!')

   NLP FEATURE EXTRACTION — Shakespeare — SUMMARY
  Total documents     : 113523
  Vocabulary (DF)     : 59034 unique words
  TF-IDF features     : 10000 hash buckets
  Word2Vec vocab      : 19796 words
  Word2Vec vector dim : 100
  Keyword searched    : "hamlet" → 27 docs

  Output saved to: output/NLP_Shakespeare/
    ├── word_doc_freq/      (DF per word)
    ├── keyword_search/     (documents with keyword)
    ├── word_vectors/       (Word2Vec embeddings)
    └── similar_words/      (Word2Vec similarities)

✅ Spark stopped. All done!
